In [0]:
CREATE OR REPLACE TABLE sb_activities (
  activity_id INT,
  user_id INT,
  activity_type STRING,
  time_spent DOUBLE,
  activity_date TIMESTAMP
);

INSERT INTO sb_activities
  VALUES
    (7274, 123, 'open', 4.5, '2022-06-22 12:00:00'),
    (2425, 123, 'send', 3.5, '2022-06-22 12:00:00'),
    (1413, 456, 'send', 5.67, '2022-06-23 12:00:00'),
    (1414, 789, 'chat', 11.0, '2022-06-25 12:00:00'),
    (2536, 456, 'open', 3.0, '2022-06-25 12:00:00');

CREATE OR REPLACE TABLE sb_age_breakdown (user_id INT, age_bucket STRING);

INSERT INTO sb_age_breakdown
  VALUES (123, '31-35'), (456, '26-30'), (789, '21-25');

with sb_age as (
  select
    sb_age.age_bucket,
    sum(
      case
        when sb_act.activity_type = 'open' then sb_act.time_spent
        else 0
      end
    ) as total_open_time,
    sum(
      case
        when sb_act.activity_type = 'send' then sb_act.time_spent
        else 0
      end
    ) as total_send_time,
    sum(sb_act.time_spent) as total_time
  from
    sb_activities sb_act
      join sb_age_breakdown sb_age
        on sb_act.user_id = sb_age.user_id
  group by
    sb_age.age_bucket
)
select
  age_bucket,
  NULLIFZERO(round((total_send_time * 100 / total_time), 2)) as send_perc,
  NULLIFZERO(round((total_open_time * 100 / total_time), 2)) as open_perc
from
  sb_age
order by
  age_bucket;